In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder \
    .appName("RideSharing_Gold") \
    .getOrCreate()

In [4]:
base_path="/content/drive/MyDrive/Ride_Sharing"

In [5]:
silver = spark.read.parquet(
    f"{base_path}/data/silver/silver_data"
)

silver.show(5)

+-------+---------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+------+-------------------+-------------------+-------------+-----------------+-------------+-----------------+------------------+
|trip_id|driver_id|pickup_location|  drop_location|distance_km|fare_amount|trip_status|    name|     city|rating|log_id|         start_time|           end_time|delay_minutes|cancellation_flag|trip_duration|completion_status|driver_performance|
+-------+---------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+------+-------------------+-------------------+-------------+-----------------+-------------+-----------------+------------------+
|     14|       99|        IT Park|        IT Park|      17.96|     232.49|  Completed|Rahul_99|    Delhi|   4.2|    14|2025-01-04 14:23:00|2025-01-04 14:32:00|           20|                0|          9.0|        Completed|              Good|
|     15|       85|     

In [6]:
total_revenue = silver.agg(
    sum("fare_amount").alias("Total_Revenue")
)

total_revenue.show()

+------------------+
|     Total_Revenue|
+------------------+
|10091.080000000004|
+------------------+



In [7]:
driver_revenue = silver.groupBy(
    "driver_id",
    "name"
).agg(
    round(sum("fare_amount"),2).alias("Total_Revenue")
).orderBy(
    col("Total_Revenue").desc()
)

driver_revenue.show()

+---------+---------+-------------+
|driver_id|     name|Total_Revenue|
+---------+---------+-------------+
|      114| Neha_114|       720.44|
|      107|Rahul_107|       530.96|
|       65|  Amit_65|       493.75|
|      102|Vikas_102|       390.52|
|      132|Priya_132|       387.74|
|       76| Priya_76|       369.47|
|       20| Vikas_20|       336.66|
|       63|  Amit_63|       334.21|
|      112| Amit_112|        326.4|
|      103|Vikas_103|       256.07|
|       18| Karan_18|       250.87|
|      110|Vikas_110|       249.72|
|       68| Arjun_68|       248.99|
|       31|  Neha_31|       242.94|
|       87| Vikas_87|       236.14|
|       99| Rahul_99|       232.49|
|       38| Sneha_38|       223.44|
|      125| Neha_125|       214.39|
|       54|Anjali_54|       212.08|
|       16| Arjun_16|       211.15|
+---------+---------+-------------+
only showing top 20 rows


In [8]:
top_drivers = driver_revenue.limit(10)

top_drivers.show()

+---------+---------+-------------+
|driver_id|     name|Total_Revenue|
+---------+---------+-------------+
|      114| Neha_114|       720.44|
|      107|Rahul_107|       530.96|
|       65|  Amit_65|       493.75|
|      102|Vikas_102|       390.52|
|      132|Priya_132|       387.74|
|       76| Priya_76|       369.47|
|       20| Vikas_20|       336.66|
|       63|  Amit_63|       334.21|
|      112| Amit_112|        326.4|
|      103|Vikas_103|       256.07|
+---------+---------+-------------+



In [9]:
cancellation = silver.groupBy(
    "completion_status"
).count()

cancellation.show()

+-----------------+-----+
|completion_status|count|
+-----------------+-----+
|        Completed|   63|
+-----------------+-----+



In [10]:
avg_duration = silver.agg(
    round(avg("trip_duration"),2).alias("Average_Trip_Duration")
)

avg_duration.show()

+---------------------+
|Average_Trip_Duration|
+---------------------+
|                33.48|
+---------------------+



In [11]:
city_revenue = silver.groupBy(
    "city"
).agg(
    round(sum("fare_amount"),2).alias("Revenue")
).orderBy(
    col("Revenue").desc()
)

city_revenue.show()

+---------+-------+
|     city|Revenue|
+---------+-------+
|    Delhi|2851.14|
|Bangalore|2535.43|
|   Mumbai|2286.38|
|     Pune|1493.44|
|Hyderabad| 924.69|
+---------+-------+



In [12]:
popular_pickup = silver.groupBy(
    "pickup_location"
).count().orderBy(
    col("count").desc()
)

popular_pickup.show()

+---------------+-----+
|pickup_location|count|
+---------------+-----+
|        IT Park|   17|
|Railway Station|   14|
|           Mall|   13|
|        Airport|   13|
|    City Center|    6|
+---------------+-----+



In [13]:
delay = silver.groupBy(
    "pickup_location"
).agg(
    round(avg("delay_minutes"),2).alias("Average_Delay")
).orderBy(
    col("Average_Delay").desc()
)

delay.show()

+---------------+-------------+
|pickup_location|Average_Delay|
+---------------+-------------+
|        IT Park|        13.88|
|        Airport|        11.85|
|           Mall|        10.69|
|    City Center|          9.5|
|Railway Station|         9.29|
+---------------+-------------+



In [14]:
windowSpec = Window.orderBy(col("fare_amount").desc())

ranked = silver.withColumn(
    "Driver_Rank",
    dense_rank().over(windowSpec)
)

ranked.select(
    "driver_id",
    "name",
    "fare_amount",
    "Driver_Rank"
).show()

+---------+---------+-----------+-----------+
|driver_id|     name|fare_amount|Driver_Rank|
+---------+---------+-----------+-----------+
|       63|  Amit_63|     334.21|          1|
|      114| Neha_114|     333.76|          2|
|      112| Amit_112|      326.4|          3|
|      107|Rahul_107|     315.07|          4|
|      102|Vikas_102|     291.12|          5|
|       18| Karan_18|     250.87|          6|
|      110|Vikas_110|     249.72|          7|
|       68| Arjun_68|     248.99|          8|
|       31|  Neha_31|     242.94|          9|
|       65|  Amit_65|     239.39|         10|
|       87| Vikas_87|     236.14|         11|
|       99| Rahul_99|     232.49|         12|
|       65|  Amit_65|     229.49|         13|
|      107|Rahul_107|     215.89|         14|
|       20| Vikas_20|     215.77|         15|
|      132|Priya_132|     213.02|         16|
|       54|Anjali_54|     212.08|         17|
|       16| Arjun_16|     211.15|         18|
|      145|Rahul_145|     204.85| 

In [15]:
performance = silver.groupBy(
    "driver_performance"
).count()

performance.show()

+------------------+-----+
|driver_performance|count|
+------------------+-----+
|         Excellent|   21|
|           Average|   25|
|              Good|   17|
+------------------+-----+



In [16]:
gold = silver.select(
    "driver_id",
    "name",
    "city",
    "fare_amount",
    "trip_duration",
    "delay_minutes",
    "completion_status",
    "driver_performance"
)

gold.show(10)

+---------+---------+---------+-----------+-------------+-------------+-----------------+------------------+
|driver_id|     name|     city|fare_amount|trip_duration|delay_minutes|completion_status|driver_performance|
+---------+---------+---------+-----------+-------------+-------------+-----------------+------------------+
|       99| Rahul_99|    Delhi|     232.49|          9.0|           20|        Completed|              Good|
|       85| Karan_85|   Mumbai|     157.98|         44.0|           19|        Completed|         Excellent|
|       65|  Amit_65|Bangalore|     239.39|         11.0|           10|        Completed|           Average|
|       80| Rohit_80|   Mumbai|     121.54|         45.0|            5|        Completed|         Excellent|
|       94|  Amit_94|     Pune|     204.54|         18.0|            4|        Completed|           Average|
|       77|Anjali_77|    Delhi|     189.57|          6.0|           13|        Completed|           Average|
|       86| Arjun_8

In [17]:
gold.write.mode("overwrite").parquet(
    f"{base_path}/data/gold/gold_data"
)

print("Gold Layer Created Successfully!")

Gold Layer Created Successfully!


In [18]:
gold_data = spark.read.parquet(
    f"{base_path}/data/gold/gold_data"
)

gold_data.show(10)

+---------+---------+---------+-----------+-------------+-------------+-----------------+------------------+
|driver_id|     name|     city|fare_amount|trip_duration|delay_minutes|completion_status|driver_performance|
+---------+---------+---------+-----------+-------------+-------------+-----------------+------------------+
|       99| Rahul_99|    Delhi|     232.49|          9.0|           20|        Completed|              Good|
|       85| Karan_85|   Mumbai|     157.98|         44.0|           19|        Completed|         Excellent|
|       65|  Amit_65|Bangalore|     239.39|         11.0|           10|        Completed|           Average|
|       80| Rohit_80|   Mumbai|     121.54|         45.0|            5|        Completed|         Excellent|
|       94|  Amit_94|     Pune|     204.54|         18.0|            4|        Completed|           Average|
|       77|Anjali_77|    Delhi|     189.57|          6.0|           13|        Completed|           Average|
|       86| Arjun_8

In [19]:
spark.stop()
print("Spark Session Closed Successfully!")

Spark Session Closed Successfully!
